In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"
 
print(os.getcwd())
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, "pyproject.toml")) and project_root != os.path.dirname(project_root):
    project_root = os.path.dirname(project_root)
os.chdir(project_root)
print(os.getcwd())

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick 
import numpy as np
from model_utils.model_config import get_model_path
from transformers import AutoTokenizer, AutoModelForCausalLM
from utils.data_util import create_data, select_split
from model_utils.base_model import BaseModel

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [ ]:
model_name_list = ["qwen3-4b", "qwen3-8b", "qwen3-14b" ,"toolace-2.5-8b", "watt-tool-8b"]

source_dir = "data/SABEval"
output_dir = "data"

In [ ]:
results = {}
for model_name in model_name_list:
    base_model = BaseModel.create(model_name)
    model = base_model.model

    results[model_name] = {
        "pair": {},
        "ground_truth": {}
    }

    add_param_num_list = [0, 1, 2, 3, 4]
    for add_param_num in add_param_num_list:
        results[model_name]["pair"][add_param_num] = create_data(base_model, data_folder=source_dir, output_dir=output_dir, mode="pair", add_param_num=add_param_num, verbose=True,sys_prompt_baseline=True)
        print(f"Model: {model_name}, Mode: pair, Add Param Num: {add_param_num}, Generated {len(results)} data points.")

    add_param_num_list = [0, 1, 2, 3, 4]
    for add_param_num in add_param_num_list:
        results[model_name]["ground_truth"][add_param_num] = create_data(base_model, data_folder=source_dir, output_dir=output_dir, mode="ground_truth", add_param_num=add_param_num, verbose=True,sys_prompt_baseline=True)
        print(f"Model: {model_name}, Mode: ground_truth, Add Param Num: {add_param_num}, Generated {len(results)} data points.")



In [ ]:
def check_tool_call(model_name, item, tokenizer=None):
    tool_call = 0
    if model_name not in ["toolace-2.5-8b", "watt-tool-8b"]:
        tool_call =  1 if item["logits_info"]["tool_call_token_rank"]==0 else 0
    else:
        if not tokenizer:
            tokenizer = AutoTokenizer.from_pretrained(get_model_path(model_name), use_fast=False)
        out_str = tokenizer.decode(item["logits_info"]["top_token_ids"][0], skip_special_tokens=True)
        if "]" not in out_str:
            if out_str.startswith("["):
                if len(out_str) == 1:
                    tool_call = 1
                elif out_str[1] == item["tool_name"][0]:
                    tool_call = 1
    return tool_call

In [ ]:
len(results["qwen3-8b"]["pair"][0])
temp_pair = select_split(results["qwen3-8b"]["pair"][0], "test")

In [ ]:
len(temp_pair)
tir = 0
for item in temp_pair:
    tir += check_tool_call("qwen3-8b", item)
tir /= len(temp_pair)

In [ ]:
for model_name in model_name_list:
    save_dir = f"results/{model_name}/main_result"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "test_all_sys_prompt_baseline.json")

    tokenizer = None
    if model_name in ["toolace-2.5-8b", "watt-tool-8b"]:
        tokenizer = AutoTokenizer.from_pretrained(get_model_path(model_name), use_fast=False)
    
    result_dict = {}
    for add_param_num in [0,1,2,3,4]:
        temp_pair = select_split(results[model_name]["pair"][add_param_num], "test")
        pair_tir = 0
        for item in temp_pair:
            pair_tir += check_tool_call(model_name, item, tokenizer)
        pair_tir /= len(temp_pair)

        temp_gt = select_split(results[model_name]["ground_truth"][add_param_num], "test")
        gt_tir = 0
        for item in temp_gt:
            gt_tir += check_tool_call(model_name, item, tokenizer)
        gt_tir /= len(temp_gt)

        result_dict[add_param_num] = {
            "pair_tir": pair_tir,
            "ground_truth_tir": gt_tir
        }
        
    with open(save_path, "w") as f:
        json.dump(result_dict, f, indent=2)
        
